## Create a Kubernetes Pod that Mounts Your PVC

In [ ]:
import os
import sys
from jinja2 import Template
import yaml

**Fill the variables below with the appropriate information**

In [5]:
sso                   = "sknnh"     # as string, replace with your SSO, all lower case
persistentVolume_name = "cloudcomp-solaiman"  # as string, replace with your pvc

In [ ]:
template = """
apiVersion: v1
kind: Pod
metadata:
  name: pod-{{sso}}-train
spec:
  automountServiceAccountToken: false
  containers:
  - name: pod-{{sso}}-train
    image: gitlab-registry.nrp-nautilus.io/skhan/dl_pytorch:4bd8b3be
    tty: true
    stdin: true
    workingDir: /data
    command: ["/bin/sh", "-c", "echo 'Im a new pod' && sleep infinity"]
    volumeMounts:
    - name: {{persistentVolume_name}}
      mountPath: /data
    resources:
      limits:
        memory: 10Gi
        cpu: "1"
      requests:
        memory: 10Gi
        cpu: "1"
  volumes:
  - name: {{persistentVolume_name}}
    persistentVolumeClaim:
      claimName: {{persistentVolume_name}}
  restartPolicy: Never
"""


# Create a Jinja2 Template object from the YAML template string
j2_template = Template(template)

# Prepare the mapping of template variables to their values
data = {
  "sso": sso,
  "persistentVolume_name": persistentVolume_name
}

# Render the template with the provided data to produce the final YAML text
output_file = j2_template.render(data)

# Write the rendered YAML to a file named using the SSO value
fileout = open("pod-{}-train.yml".format(sso), "w")
fileout.write(output_file)
fileout.close()

# This script generates a Kubernetes Pod YAML file using Jinja2 templating.
# You can apply the generated file with: kubectl apply -f pod-<sso>-train.yml